Cell 1 — Re-parse XML with entity types

In [23]:
import xml.etree.ElementTree as ET
import os
import pandas as pd

def parse_ddi_xml(folder_path):
    data = []
    
    for root_dir, dirs, files in os.walk(folder_path):
        for filename in files:
            if not filename.endswith(".xml"):
                continue
                
            filepath = os.path.join(root_dir, filename)
            tree = ET.parse(filepath)
            root = tree.getroot()
            
            for sentence in root.findall(".//sentence"):
                sentence_text = sentence.get("text", "")
                
                # Get all drug entities WITH their types
                entities = {}
                for entity in sentence.findall("entity"):
                    entities[entity.get("id")] = {
                        "text": entity.get("text"),
                        "type": entity.get("type", "drug")  # drug, brand, group, drug_n
                    }
                
                # Get all pairs
                for pair in sentence.findall("pair"):
                    e1_id = pair.get("e1")
                    e2_id = pair.get("e2")
                    ddi = pair.get("ddi")
                    ddi_type = pair.get("type", "NO-REL")
                    
                    if e1_id not in entities or e2_id not in entities:
                        continue
                    
                    e1_text = entities[e1_id]["text"]
                    e2_text = entities[e2_id]["text"]
                    e1_type = entities[e1_id]["type"]  # NEW
                    e2_type = entities[e2_id]["type"]  # NEW
                    
                    relation = "NO-REL" if ddi == "false" else (ddi_type if ddi_type else "NO-REL")
                    
                    data.append({
                        "sentence": sentence_text,
                        "entity1": e1_text,
                        "entity2": e2_text,
                        "type1": e1_type,   # NEW
                        "type2": e2_type,   # NEW
                        "relation": relation
                    })
    
    return pd.DataFrame(data)

# Re-parse with types
train_df = parse_ddi_xml("../data/ddi_corpus/Train")
test_df = parse_ddi_xml("../data/ddi_corpus/Test")

print("Train:", len(train_df))
print("Test:", len(test_df))
print("\nEntity type distribution:")
print(train_df["type1"].value_counts())
print("\nSample with types:")
print(train_df[["entity1", "type1", "entity2", "type2", "relation"]].head(5))

Train: 27792
Test: 6657

Entity type distribution:
type1
drug      17393
group      7134
brand      2763
drug_n      502
Name: count, dtype: int64

Sample with types:
           entity1   type1     entity2   type2 relation
0  contortrostatin  drug_n  echistatin  drug_n   NO-REL
1  contortrostatin  drug_n  flavoridin  drug_n   NO-REL
2       echistatin  drug_n  flavoridin  drug_n   NO-REL
3  contortrostatin  drug_n  echistatin  drug_n   NO-REL
4  contortrostatin  drug_n  flavoridin  drug_n   NO-REL


Cell 2 — Updated typed entity marker function

In [24]:
def add_typed_entity_markers(sentence, e1, t1, e2, t2):
    # Create typed tags e.g. [DRUG-START], [BRAND-END]
    e1_start = f"[{t1.upper()}-START]"
    e1_end = f"[{t1.upper()}-END]"
    e2_start = f"[{t2.upper()}-START]"
    e2_end = f"[{t2.upper()}-END]"
    
    # Sort by length to avoid partial replacement
    entities = sorted(
        [(e1, e1_start, e1_end), (e2, e2_start, e2_end)],
        key=lambda x: len(x[0]),
        reverse=True
    )
    
    marked = sentence
    for entity, start_tag, end_tag in entities:
        marked = marked.replace(entity, f"{start_tag}{entity}{end_tag}", 1)
    
    return marked

# Test it
print("=== Typed Entity Marker Examples ===\n")
sample_rows = train_df[train_df["relation"] != "NO-REL"].head(5)
for _, row in sample_rows.iterrows():
    marked = add_typed_entity_markers(
        row["sentence"], 
        row["entity1"], row["type1"],
        row["entity2"], row["type2"]
    )
    print(f"Relation: {row['relation']}")
    print(f"Type1: {row['type1']}, Type2: {row['type2']}")
    print(f"Marked: {marked[:180]}")
    print()

=== Typed Entity Marker Examples ===

Relation: effect
Type1: drug_n, Type2: drug_n
Marked: [DRUG_N-START]Echistatin[DRUG_N-END] alone had no effect on tyrosine phosphorylation in T24 cells, but dose-dependently inhibits the effects of [DRUG_N-START]contortrostatin[DRUG_N

Relation: effect
Type1: drug_n, Type2: drug_n
Marked: [DRUG_N-START]Flavoridin[DRUG_N-END] alone was found to have no effect on CAS, but can completely block [DRUG_N-START]contortrostatin[DRUG_N-END]-induced phosphorylation of this pr

Relation: effect
Type1: drug, Type2: drug_n
Marked: Prior administration of [DRUG-START]4-methylpyrazole[DRUG-END] (90 mg kg(-1) body weight) was shown to prevent the conversion of [DRUG_N-START]1,3-difluoro-2-propanol[DRUG_N-END] (

Relation: effect
Type1: drug, Type2: drug_n
Marked: We conclude that the prophylactic and antidotal properties of [DRUG-START]4-methylpyrazole[DRUG-END] seen in animals treated with [DRUG_N-START]1,3-difluoro-2-propanol[DRUG_N-END] 

Relation: effect
Type1

Cell 3 — Apply typed markers to full dataset

In [25]:
def prepare_re_sample(row):
    return add_typed_entity_markers(
        str(row["sentence"]),
        str(row["entity1"]), str(row["type1"]),
        str(row["entity2"]), str(row["type2"])
    )

train_df["marked_sentence"] = train_df.apply(prepare_re_sample, axis=1)
test_df["marked_sentence"] = test_df.apply(prepare_re_sample, axis=1)

label2id = {"NO-REL": 0, "effect": 1, "mechanism": 2, "advise": 3, "int": 4}
train_df["label"] = train_df["relation"].map(label2id)
test_df["label"] = test_df["relation"].map(label2id)

print("Typed markers applied!")
print("\nSample marked sentence:")
print(train_df["marked_sentence"].iloc[100])

Typed markers applied!

Sample marked sentence:
The lower rate of absorption in the groups receiving 446 mg [DRUG-START][DRUG-START]Fe[DRUG-END][DRUG-END] instead of 48 mg of Fe per kg diet resulted in a decreased renal excretion of cobalt. 


Cell 4 — Downsample NO-REL

In [26]:
# No downsampling - using full dataset with smoothed weights instead
print("Using full dataset without downsampling")
print(f"Train size: {len(train_df)}")
print("\nClass Distribution:")
print(train_df["relation"].value_counts())

Using full dataset without downsampling
Train size: 27792

Class Distribution:
relation
NO-REL       23772
effect        1687
mechanism     1319
advise         826
int            188
Name: count, dtype: int64


Cell 5 — Add all typed special tokens to tokenizer

In [27]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

special_tokens = [
    "[DRUG-START]", "[DRUG-END]",
    "[BRAND-START]", "[BRAND-END]",
    "[GROUP-START]", "[GROUP-END]",
    "[DRUG_N-START]", "[DRUG_N-END]"
]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print("Vocabulary size:", len(tokenizer))
print("Special tokens added:", special_tokens)

# Verify
test_text = "[DRUG-START]Warfarin[DRUG-END] increases effect of [BRAND-START]Aspirin[BRAND-END]"
tokens = tokenizer.tokenize(test_text)
print("\nTokenized:", tokens)

Vocabulary size: 29004
Special tokens added: ['[DRUG-START]', '[DRUG-END]', '[BRAND-START]', '[BRAND-END]', '[GROUP-START]', '[GROUP-END]', '[DRUG_N-START]', '[DRUG_N-END]']

Tokenized: ['[DRUG-START]', 'war', '##fari', '##n', '[DRUG-END]', 'increases', 'effect', 'of', '[BRAND-START]', 'as', '##pi', '##rin', '[BRAND-END]']



Cell 6 — Recalculate class weights on balanced dataset

In [28]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import json

id2label = {v: k for k, v in label2id.items()}
labels_array = np.array(train_df["label"].tolist())

# Standard class weights
raw_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_array),
    y=labels_array
)

# Apply square root smoothing
smoothed_weights = np.sqrt(raw_weights)

print("=== Smoothed Class Weights ===\n")
for idx in range(len(smoothed_weights)):
    count = len(train_df[train_df["label"] == idx])
    print(f"{id2label[idx]:12} → raw: {raw_weights[idx]:6.2f}  smoothed: {smoothed_weights[idx]:.4f}  ({count} samples)")

=== Smoothed Class Weights ===

NO-REL       → raw:   0.23  smoothed: 0.4836  (23772 samples)
effect       → raw:   3.29  smoothed: 1.8152  (1687 samples)
mechanism    → raw:   4.21  smoothed: 2.0528  (1319 samples)
advise       → raw:   6.73  smoothed: 2.5941  (826 samples)
int          → raw:  29.57  smoothed: 5.4375  (188 samples)


Cell 7 — Save everything

In [29]:
train_df[["marked_sentence", "relation", "label"]].to_csv(
    "../data/ddi_train_marked.csv", index=False
)
test_df[["marked_sentence", "relation", "label"]].to_csv(
    "../data/ddi_test_marked.csv", index=False
)

# Save smoothed weights
weights_dict = {id2label[i]: float(smoothed_weights[i]) for i in range(len(smoothed_weights))}
with open("../data/class_weights.json", "w") as f:
    json.dump(weights_dict, f, indent=2)

print("Saved with smoothed weights!")
print("Weights:", weights_dict)

Saved with smoothed weights!
Weights: {'NO-REL': 0.4835507236811244, 'effect': 1.815170216927239, 'mechanism': 2.052827706388249, 'advise': 2.594089015593575, 'int': 5.4374587305843995}
